# DFI Quant — Feature Dashboard

Vue centralisée de toutes les features daily.
Lit depuis `reports/ic/*.json` et `reports/qa/*.md` — aucun recalcul.

Pour mettre à jour : relancer les agents puis ré-exécuter ce notebook.

In [ ]:
import pathlib, json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

ROOT    = pathlib.Path('..').resolve()
IC_DIR  = ROOT / 'reports' / 'ic'
QA_DIR  = ROOT / 'reports' / 'qa'

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
print('IC reports :', sorted(p.stem for p in IC_DIR.glob('*.json')))
print('QA reports :', sorted(p.stem for p in QA_DIR.glob('*.md')))

In [ ]:
# ── Chargement des résultats IC ───────────────────────────────────────────────
ic_results = {}
for path in sorted(IC_DIR.glob('*.json')):
    data = json.loads(path.read_text())
    ic_results[data['feature_id']] = data

# ── Chargement des résultats QA ───────────────────────────────────────────────
qa_results = {}
for path in sorted(QA_DIR.glob('*.md')):
    text = path.read_text()
    qa_results[path.stem] = '✅ PASS' in text

print(f'{len(ic_results)} features IC chargées')
print(f'{len(qa_results)} rapports QA chargés')

## 1. Leaderboard

In [ ]:
rows = []
for fid, data in ic_results.items():
    ic_h  = {int(k): v for k, v in data['ic_mean'].items()}
    icir_h = {int(k): v for k, v in data['icir'].items()}
    ic_ortho_h = {int(k): v for k, v in data.get('ic_mean_ortho', {}).items()}

    best_h   = max(ic_h, key=lambda h: abs(ic_h[h]) if ic_h[h] != 'nan' else 0)
    icir_1   = icir_h.get(1, float('nan'))
    icir_1   = float(icir_1) if icir_1 != 'nan' else float('nan')
    ic_1     = float(ic_h.get(1, float('nan'))) if ic_h.get(1) != 'nan' else float('nan')
    ic_ortho = float(ic_ortho_h.get(1, float('nan'))) if ic_ortho_h.get(1, 'nan') != 'nan' else float('nan')

    regime_data = data.get('regime', {}).get('1', {})
    best_regime = max(regime_data.items(),
                      key=lambda kv: abs(kv[1].get('mean_ic', 0)),
                      default=('?', {}))[0] if regime_data else '?'

    rows.append({
        'Feature':       fid,
        'QA':            '✅' if qa_results.get(fid, False) else '❌',
        'IC 1d':         round(ic_1, 4),
        'ICIR 1d':       round(icir_1, 3),
        'IC ortho 1d':   round(ic_ortho, 4),
        'Meilleur H':    f'{best_h}d',
        'Régime fort':   best_regime,
        'Exploitable':   '⭐' if abs(icir_1) >= 0.10 else '',
    })

leaderboard = (pd.DataFrame(rows)
               .sort_values('ICIR 1d', key=abs, ascending=False)
               .reset_index(drop=True))
leaderboard.index += 1

print('FEATURE LEADERBOARD\n')
print(leaderboard.to_string())
print('\n⭐ = |ICIR| ≥ 0.10 — exploitable en production')

## 2. Decay curves — IC moyen par horizon

In [ ]:
n = len(ic_results)
cols = min(n, 3)
rows_n = (n + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(5 * cols, 4 * rows_n))
axes = np.array(axes).flatten()

for ax, (fid, data) in zip(axes, ic_results.items()):
    ic_h = {int(k): v for k, v in data['ic_mean'].items()}
    horizons = sorted(ic_h.keys())
    means = [float(ic_h[h]) if ic_h[h] != 'nan' else 0 for h in horizons]
    colors = ['tab:green' if m >= 0 else 'tab:red' for m in means]
    ax.bar([f'{h}d' for h in horizons], means, color=colors, alpha=0.8)
    ax.axhline(0, color='k', lw=0.6)
    qa_flag = '✅' if qa_results.get(fid, False) else '❌'
    icir_1 = data['icir'].get('1', 'nan')
    icir_1 = float(icir_1) if icir_1 != 'nan' else float('nan')
    ax.set_title(f'{fid}  {qa_flag}\nICIR 1d = {icir_1:+.3f}', fontsize=9)
    ax.set_ylabel('IC moyen')

for ax in axes[n:]:
    ax.set_visible(False)

fig.suptitle('Decay curves — IC moyen par horizon', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. IC Raw vs Ortho — apport unique de chaque feature

In [ ]:
fids   = list(ic_results.keys())
raw    = [float(ic_results[f]['ic_mean'].get('1', 0))        if ic_results[f]['ic_mean'].get('1') != 'nan' else 0 for f in fids]
ortho  = [float(ic_results[f].get('ic_mean_ortho', {}).get('1', 0)) if ic_results[f].get('ic_mean_ortho', {}).get('1', 'nan') != 'nan' else 0 for f in fids]

x = np.arange(len(fids))
w = 0.35
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - w/2, raw,   w, label='Raw',   color='steelblue',  alpha=0.8)
ax.bar(x + w/2, ortho, w, label='Ortho', color='darkorange', alpha=0.8)
ax.axhline(0, color='k', lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(fids, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('IC moyen — horizon 1d')
ax.set_title('Raw vs Ortho IC (mom_20d + rv_30d retirés) — horizon 1d')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Régimes — IC en haute vs basse volatilité BTC

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(fids))
w = 0.35

high_ic = []
low_ic  = []
for f in fids:
    regime = ic_results[f].get('regime', {}).get('1', {})
    high_ic.append(float(regime.get('high_vol', {}).get('mean_ic', 0) or 0))
    low_ic.append(float(regime.get('low_vol',  {}).get('mean_ic', 0) or 0))

ax.bar(x - w/2, high_ic, w, label='Haute vol', color='tab:orange', alpha=0.8)
ax.bar(x + w/2, low_ic,  w, label='Basse vol',  color='tab:blue',   alpha=0.8)
ax.axhline(0, color='k', lw=0.6)
ax.set_xticks(x)
ax.set_xticklabels(fids, rotation=20, ha='right', fontsize=9)
ax.set_ylabel('IC moyen — horizon 1d')
ax.set_title('IC par régime de volatilité BTC — horizon 1d')
ax.legend()
plt.tight_layout()
plt.show()

## 5. Matrice de corrélation entre features (collinéarité)

In [ ]:
import importlib, yaml

sys.path.insert(0, str(ROOT))

def load_ohlcv(sym, exchange='binance-futures'):
    base = ROOT / 'data' / 'raw' / f'exchange={exchange}' / 'data_type=ohlcv_1d' / f'symbol={sym}'
    parts = sorted(base.rglob('part-*.parquet'))
    if not parts:
        return pd.DataFrame()
    df = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True)
    df['date'] = pd.to_datetime(df['ts_open'], unit='us', utc=True).dt.normalize()
    return df.sort_values('date').set_index('date')

with open(ROOT / 'configs' / 'features.yaml') as f:
    registry = yaml.safe_load(f)['features']
with open(ROOT / 'configs' / 'universe.yaml') as f:
    assets = [a['symbol'] for a in yaml.safe_load(f)['assets']]

ohlcv_features = [feat for feat in registry
                  if feat.get('source') == 'ohlcv_1d' and feat['id'] in ic_results]

prices = {s: load_ohlcv(s) for s in assets}

# Compute signals pooled across assets
pooled = {}
for feat in ohlcv_features:
    fid = feat['id']
    try:
        mod = importlib.import_module(f'dfi_features.{fid}')
        parts = [mod.compute(df, feat.get('params', {})).rename(fid)
                 for df in prices.values() if not df.empty]
        if parts:
            pooled[fid] = pd.concat(parts)
    except Exception:
        pass

if len(pooled) >= 2:
    corr_df = pd.DataFrame(pooled).corr(method='spearman')
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(corr_df.values, vmin=-1, vmax=1, cmap='RdYlGn', aspect='auto')
    labels = list(corr_df.columns)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    for i in range(len(labels)):
        for j in range(len(labels)):
            v = corr_df.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        fontsize=8, color='black' if abs(v) < 0.6 else 'white')
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title('Matrice de corrélation Spearman (poolée cross-assets)\n|corr| > 0.70 = collinéaire', fontsize=10)
    plt.tight_layout()
    plt.show()
    print('\nPaires collinéaires (|corr| > 0.70) :')
    for i, f1 in enumerate(labels):
        for f2 in labels[i+1:]:
            v = corr_df.loc[f1, f2]
            if abs(v) > 0.70:
                print(f'  {f1} ↔ {f2} : {v:+.3f}')
else:
    print('Pas assez de features pour calculer la corrélation.')

## 6. Résumé QA

In [ ]:
print('ÉTAT QA PAR FEATURE\n')
all_features = sorted(set(list(ic_results.keys()) + list(qa_results.keys())))
for fid in all_features:
    ic_ok  = '✅' if fid in ic_results else '❌'
    qa_ok  = '✅' if qa_results.get(fid, False) else '❌ (pas de rapport)'
    print(f'  {fid:<30}  IC: {ic_ok}  QA: {qa_ok}')